# 05 — Autres protocoles : cohérence, HRR, dérive, VO2max

Ce notebook présente les quatre protocoles complémentaires.
Les données RR sont synthétiques (physiologiquement réalistes) car ces protocoles
nécessitent des conditions d'enregistrement spécifiques (effort, respiration guidée).

| Protocole | Condition | Indicateur principal |
|---|---|---|
| Cohérence cardiaque | Respiration à 6 cycles/min | `coherence_score` (% puissance résonance) |
| HRR | Post-effort maximal | `hrr_60` — chute HR à 60 s (Cole 1999) |
| Dérive cardiaque | Effort constant prolongé | `drift_rate` (bpm/min) |
| VO2max | Repos + HRmax connu | `vo2max_uth`, `vo2max_esco_flatt`, `vo2max_ln_rmssd` |

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from cardiolab.analytics import (
    coherence_score_100,
    drift_score,
    hrr_score,
    vo2max_score,
)
from cardiolab.protocols import (
    cardiac_coherence,
    cardiac_drift,
    heart_rate_recovery,
    vo2max_from_hrv,
)
from cardiolab.signals.rr import RRSeries

rng = np.random.default_rng(42)
print("✓  OK")

## 1 — Cohérence cardiaque

La **cohérence cardiaque** est atteinte lorsque la respiration à **0.1 Hz** (6 cycles/min)
synchronise la variabilité sinusale avec le baroréflexe, créant une oscillation maximale
et régulière à la fréquence de résonance cardiaque.

**Protocole 5-5 :** 5 s inspiration / 5 s expiration, pendant 5 minutes.

In [ ]:
from cardiolab.visualization.coherence_plots import (
    plot_coherence_psd,
)

# Générer un signal de cohérence : oscillation sinusoïdale à 0.1 Hz + bruit faible
t = np.linspace(0, 300, 1800)  # 300 s à ~6 Hz
rr_vals = 800 + 80 * np.sin(2 * np.pi * 0.1 * t) + rng.normal(0, 8, len(t))
rr_coh = RRSeries(rr_vals.tolist())

result_coh = cardiac_coherence(rr_coh)
score_coh = coherence_score_100(result_coh.coherence_score)

print(f"Cohérence brute : {result_coh.coherence_score:.1f} %")
print(f"Fréquence de résonance : {result_coh.resonance_freq:.3f} Hz")
print(f"Score [0–100] : {score_coh:.0f}")
interp = (
    "Bonne"
    if result_coh.coherence_score >= 60
    else "Modérée"
    if result_coh.coherence_score >= 40
    else "Faible"
)
print(f"Niveau : {interp}")

fig = plot_coherence_psd(rr_coh, result_coh)
plt.show()

## 2 — Heart Rate Recovery (HRR)

La **récupération de la fréquence cardiaque** après un effort maximal est un marqueur
de mortalité cardiovasculaire (Cole et al. 1999, *NEJM*). La chute de HR à 60 secondes
(HRR1) est le paramètre le plus validé :

| HRR1 (bpm) | Catégorie |
|---|---|
| ≥ 25 | Excellent — risque très faible |
| 20–24 | Bon |
| 12–19 | Normal |
| < 12 | **Altéré** — risque cardiovasculaire élevé |

In [ ]:
from cardiolab.visualization.hrr_plots import plot_hrr_curve, plot_hrr_gauge

# Signal de récupération : HR chute de 180 bpm vers 120 bpm exponentiellement
t_hrr = np.linspace(0, 180, 900)
hr_hrr = 120 + 60 * np.exp(-t_hrr / 40) + rng.normal(0, 3, len(t_hrr))
rr_hrr = RRSeries((60_000 / hr_hrr).tolist())

result_hrr = heart_rate_recovery(rr_hrr)
score_hrr = hrr_score(result_hrr.hrr_60)

print(f"HR au pic    : {result_hrr.hr_peak:.0f} bpm")
print(f"HR à 60 s    : {result_hrr.hr_at_60s:.0f} bpm")
print(f"HRR1 (60 s)  : {result_hrr.hrr_60:.0f} bpm — {result_hrr.hrr_60_category}")
print(f"HRR2 (120 s) : {result_hrr.hrr_120:.0f} bpm — {result_hrr.hrr_120_category}")
print(f"Score        : {score_hrr:.0f}/100")

fig = plot_hrr_curve(rr_hrr, result_hrr)
plt.show()

fig = plot_hrr_gauge(result_hrr)
plt.show()

## 3 — Dérive cardiaque

La **dérive cardiaque** (cardiac drift) est l'augmentation progressive de la FC
pendant un exercice constant. Elle traduit une déshydratation, une thermorégulation
insuffisante ou une fatigue musculaire.

In [ ]:
from cardiolab.visualization.drift_plots import plot_drift_curve

# Signal de dérive : HR augmente de 140 à 162 bpm sur 20 min (dérive ~1.1 bpm/min)
t_drift = np.linspace(0, 1200, 3000)  # 20 min
hr_drift = 140 + 1.1 * (t_drift / 60) + rng.normal(0, 4, len(t_drift))
rr_drift = RRSeries((60_000 / hr_drift).tolist())

result_drift = cardiac_drift(rr_drift, window_sec=60.0)
score_dr = drift_score(result_drift.drift_rate)

print(f"Taux de dérive : {result_drift.drift_rate:.2f} bpm/min")
print(f"Magnitude      : {result_drift.drift_magnitude:.1f} bpm")
print(f"R²             : {result_drift.r_squared:.3f}")
print(f"Catégorie      : {result_drift.interpretation}")
print(f"Score          : {score_dr:.0f}/100")

fig = plot_drift_curve(rr_drift, result_drift)
plt.show()

## 4 — Estimation VO2max

Trois modèles complémentaires, tous basés sur un enregistrement HRV de repos :

| Modèle | Formule | Précision |
|---|---|---|
| **Uth (2004)** | `15.3 × (HRmax / HRrest)` | ±10–15 % |
| **Esco-Flatt (2014)** | `18.37 + 0.054 × RMSSD` | ±7–12 % |
| **ln-RMSSD** | `24.89 + 5.97 × ln(RMSSD)` | ±7–12 % |

In [ ]:
from pathlib import Path

import cardiolab
from cardiolab.sensors_tools.polar import parse_rr_file
from cardiolab.visualization.vo2max_plots import (
    plot_vo2max_comparison,
    plot_vo2max_gauge,
)

DATASETS_DIR = Path(cardiolab.__file__).parent / "datasets"

# Utiliser une vraie session repos pour VO2max
raw = parse_rr_file(sorted((DATASETS_DIR / "raw" / "resting").glob("*.txt"))[0])
rr_rest = RRSeries(raw["rr_intervals"]).remove_outliers()

result_vo2 = vo2max_from_hrv(rr_rest, hr_max=185.0, hr_rest=rr_rest.mean_hr)
score_vo2 = vo2max_score(result_vo2.vo2max_esco_flatt)

print(
    f"Modèle Uth         : {result_vo2.vo2max_uth:.1f} mL/kg/min"
    if result_vo2.vo2max_uth
    else "Modèle Uth         : (HRmax requis)"
)
print(f"Modèle Esco-Flatt  : {result_vo2.vo2max_esco_flatt:.1f} mL/kg/min")
print(f"Modèle ln-RMSSD    : {result_vo2.vo2max_ln_rmssd:.1f} mL/kg/min")
print(f"Catégorie ACSM     : {result_vo2.fitness_category}")
print(f"Score              : {score_vo2:.0f}/100")

fig = plot_vo2max_comparison(result_vo2)
plt.show()

fig = plot_vo2max_gauge(result_vo2)
plt.show()